# ④ 発展デモ：グループ分け・次元削減・回帰予測・因果推論
**岡山県 高校教員向け データサイエンス・ハンズオン（発展／デモ）**

このノートは **上から順に実行するだけ** で、機械学習の4つのテーマを「見て確認」できます。
（手を動かす演習ではなく、講師のデモ／自習の確認用です）
1. **クラスタリング（K-means）**：似たもの同士をグループに分ける
2. **主成分分析（PCA）**：たくさんの変数を少ない軸に要約して見える化する
3. **回帰分析と予測（重回帰）**：複数の要因から結果を予測する
4. **因果推論の入り口**：「相関」と「因果」を区別し、交絡を調整する

> 使うライブラリは `scikit-learn`（Colabに最初から入っています）。


## 0. 準備（実行）

In [ ]:
# === データ読み込みの準備（このセルを最初に実行）===
import pandas as pd, io, os
BASE_URL = "https://raw.githubusercontent.com/yasuyuki-nogami/ds-handson/main/data/"

def load(name):
    if os.path.exists("data/" + name):
        return pd.read_csv("data/" + name)
    if BASE_URL:
        try:
            return pd.read_csv(BASE_URL + name)
        except Exception:
            pass
    from google.colab import files
    print(name, " をアップロードしてください")
    up = files.upload(); key = list(up.keys())[0]
    return pd.read_csv(io.BytesIO(up[key]))
print("準備OK")


In [ ]:
!pip -q install japanize-matplotlib
import matplotlib.pyplot as plt
import japanize_matplotlib
import numpy as np
print("準備OK")


---
# 1. クラスタリング（K-means）：47都道府県をグループ分け
「人口・面積・人口密度」の似ている都道府県を、コンピュータが自動でグループ分けします。
正解ラベルを与えずにデータの構造を見つけるので **教師なし学習** と呼びます。


In [ ]:
pref = load("prefecture.csv")

# 分析に使う3つの特徴量（数値の列）を選ぶ
X = pref[["population_2020", "area_km2", "density_per_km2"]]

# --- なぜ「標準化」するのか？ -------------------------------------------
# 人口は数百万、面積は数千、密度は数百…と桁がバラバラ。
# そのまま距離を測ると「数字が大きい列（人口）」ばかりが効いてしまう。
# 標準化＝各列を「平均0・ばらつき1」にそろえる作業。これで公平に比較できる。
# ----------------------------------------------------------------------
from sklearn.preprocessing import StandardScaler
Xs = StandardScaler().fit_transform(X)
print("標準化後の形（行=都道府県, 列=特徴量）:", Xs.shape)


In [ ]:
from sklearn.cluster import KMeans

# n_clusters=3 → 3つのグループに分ける、という指定
# random_state=0 は「毎回同じ結果」にするためのおまじない（再現性のため）
km = KMeans(n_clusters=3, n_init=10, random_state=0)
pref["cluster"] = km.fit_predict(Xs)   # 各都道府県に 0/1/2 のグループ番号が付く

print("各グループの県数:")
print(pref["cluster"].value_counts().sort_index())
pref[["prefecture", "population_2020", "density_per_km2", "cluster"]].head(10)


In [ ]:
# グループごとの平均を見ると、各グループの「性格」が読み取れる
# （例：人口も密度も大きい＝大都市圏、面積が広く密度が低い＝地方 など）
pref.groupby("cluster")[["population_2020","area_km2","density_per_km2"]].mean().round(0)


In [ ]:
# 散布図で色分け（人口 × 人口密度）：同じ色＝同じグループ
plt.figure(figsize=(7,5))
for c in sorted(pref["cluster"].unique()):
    sub = pref[pref["cluster"] == c]
    plt.scatter(sub["population_2020"]/10000, sub["density_per_km2"], label=f"グループ{c}")
plt.xlabel("人口（万人）"); plt.ylabel("人口密度（人/km²）")
plt.title("K-meansによる都道府県のグループ分け")
plt.legend(); plt.show()


In [ ]:
# 各グループにどの都道府県が入ったか
for c in sorted(pref["cluster"].unique()):
    names = pref.loc[pref["cluster"]==c, "prefecture"].tolist()
    print(f"■ グループ{c}（{len(names)}件）:", "、".join(names))


**学びのポイント**
- クラスタリングは「正解ラベルなし」でデータの構造（似た者どうし）を見つける手法。
- 距離の計算では**標準化**が超重要（単位の違いに引っ張られないため）。
- グループ数（`n_clusters`）は人間が決める。目的に応じて2〜4などを試すのが普通。


---
# 2. 主成分分析（PCA）：たくさんの変数を「2つの軸」に要約する
上のクラスタリングは3つの変数を使いました。でも変数が5個・10個…と増えると、
散布図では見きれません。そこで **PCA（主成分分析／Principal Component Analysis）** の出番です。

**PCAとは**：たくさんの変数の情報を、なるべく失わずに **少ない数の新しい軸（主成分）** にまとめ直す
「次元削減」の手法です。3次元→2次元にすれば、1枚の散布図で全体を見渡せます。


In [ ]:
from sklearn.decomposition import PCA

# 3つの特徴量（標準化済みの Xs）を、2つの主成分（PC1, PC2）に圧縮する
pca = PCA(n_components=2)
pcs = pca.fit_transform(Xs)   # 各都道府県が (PC1, PC2) の2つの数で表される

# --- 寄与率（explained_variance_ratio_）とは？ -------------------------
# 「元のデータのばらつき（情報量）を、その主成分がどれだけ説明できているか」の割合。
# 例）PC1=0.6 なら、PC1だけで全情報の60%を説明できている、という意味。
# 累積が高い（例：2軸で0.9超）ほど、2次元に落としても情報の損失が少ない。
# ----------------------------------------------------------------------
print("各主成分の寄与率:", pca.explained_variance_ratio_.round(3))
print("累積寄与率(PC1+PC2):", pca.explained_variance_ratio_.cumsum().round(3)[-1])


In [ ]:
# --- 主成分の「意味」を読む：loadings（各元変数の効き方）--------------
# 主成分は元の変数の重み付き合計。重み(loadings)を見ると各軸の意味が分かる。
loadings = pd.DataFrame(
    pca.components_.T,
    index=["人口", "面積", "人口密度"],
    columns=["PC1", "PC2"]
)
print("各主成分への元変数の寄与（loadings）:")
print(loadings.round(2))
print("\n→ 絶対値が大きい変数ほど、その主成分の意味を強く決めている")


In [ ]:
# PCAで得た2軸(PC1, PC2)で散布図。色はK-meansのグループ。
# 3変数の情報を1枚の図に「要約」して見られるのがPCAの利点。
plt.figure(figsize=(7,6))
for c in sorted(pref["cluster"].unique()):
    m = pref["cluster"] == c
    plt.scatter(pcs[m, 0], pcs[m, 1], label=f"グループ{c}")

# 特徴的な都道府県だけ名前を表示（図が名前だらけにならないように）
for name in ["東京都", "北海道", "大阪府", "神奈川県", "鳥取県"]:
    if name in pref["prefecture"].values:
        i = pref.index[pref["prefecture"] == name][0]
        plt.annotate(name, (pcs[i, 0], pcs[i, 1]))

plt.xlabel("第1主成分 PC1"); plt.ylabel("第2主成分 PC2")
plt.title("PCAで2次元に要約した47都道府県（色=K-meansのグループ）")
plt.legend(); plt.tight_layout(); plt.show()


**学びのポイント**
- PCAは **次元削減**：多くの変数を、情報をできるだけ保ったまま少ない軸に要約する。
- **寄与率**で「その軸がどれだけ情報を持つか」、**loadings**で「その軸が何を表すか」を読む。
- K-means（分ける）と PCA（見える化する）は相性がよく、**分けた結果を2次元で確認**できる。
- 実データでは変数が多いほどPCAの価値が上がる（今回は3変数の入門例）。


---
# 3. 回帰分析と予測（重回帰）
複数の要因から「テストの点数」を予測するモデルを作ります。
※ `class_sample.csv` は練習用の**架空データ**です。


In [ ]:
df = load("class_sample.csv")

# 説明変数（原因になりそうな要因）と、目的変数（予測したいもの）を決める
features = ["study_min", "sleep_hours", "breakfast", "smartphone_hours"]
X = df[features]
y = df["test_score"]

# --- なぜ学習用とテスト用に分ける？ ------------------------------------
# 同じデータで学習・採点すると「丸暗記」でも高得点に見えてしまう。
# 一部を「テスト用」に取り分け、学習に使っていないデータで実力を測る。
# ----------------------------------------------------------------------
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)
print("学習用", X_train.shape, " テスト用", X_test.shape)


In [ ]:
from sklearn.linear_model import LinearRegression
model = LinearRegression().fit(X_train, y_train)

# 係数 = 「その要因が1単位増えると点数が何点変わるか」（プラスは上げる/マイナスは下げる要因）
coef = pd.Series(model.coef_, index=features).round(2)
print("係数（1単位あたり点数がどれだけ変わるか）")
print(coef)
print("切片:", round(model.intercept_, 1))


In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error
pred = model.predict(X_test)
# R^2: 1に近いほど当てはまり良い / MAE: 予測が平均何点ずれるか
print("R^2（1に近いほど良い）:", round(r2_score(y_test, pred), 3))
print("平均絶対誤差（点）:", round(mean_absolute_error(y_test, pred), 2))

# 予測 vs 実際 を散布図で（点が対角線に近いほど良い予測）
plt.figure(figsize=(5,5))
plt.scatter(y_test, pred, alpha=0.4)
plt.plot([0,100],[0,100], "r--")
plt.xlabel("実際の点数"); plt.ylabel("予測した点数"); plt.title("予測の精度")
plt.show()


In [ ]:
# 学習したモデルで、新しい生徒の点数を予測してみる
new_student = pd.DataFrame([{
    "study_min": 90, "sleep_hours": 7.0, "breakfast": 1, "smartphone_hours": 1.5
}])
print("この生徒の予測点数:", round(model.predict(new_student)[0], 1))


**学びのポイント**
- 重回帰は「複数の要因」から結果を予測し、各要因の効き方（係数）も分かる。
- **学習用/テスト用に分ける**ことで、予測の"本当の実力"を測れる。
- 係数はあくまで「このデータ上の関係」。因果とは限らない（→ 次の章へ）。


---
# 4. 因果推論の入り口：「本当に効果があるの？」を確かめる

## そもそも「因果推論」って？
データを見て「Aが多いとBも多い」と分かっても、**「AのおかげでBになった」とは限りません。**
"本当に効果があるのか（原因→結果の関係があるのか）"を慎重に確かめる考え方を **因果推論** といいます。

### 身近なたとえ話
- 「アイスがよく売れる日は、水の事故も多い」というデータがあったとします。
- では **アイスが事故の原因**でしょうか？ → もちろん違いますね。
- 本当の犯人は **「暑さ」**。暑い日はアイスも売れるし、水遊びも増える。
- このように、**2つのことの両方に影響する隠れた要因**を **交絡（こうらく）** と呼びます。
- 交絡があると、見かけ上の関係が「効果」だと勘違いしてしまいます。

これから、この「交絡」を身近な例で体験します。


## 今回の題材：「塾に通うと成績は上がる？」
塾に通っている子は、通っていない子より点数が高いかもしれません。
でも、塾に通う子は **もともと勉強熱心な家庭** が多いかも。だとすると、点数が高い理由が
「塾のおかげ」なのか「家庭が熱心だから」なのか、区別がつきません（＝これが交絡）。

そこで **答えをこちらで決めた"練習用の作りデータ"** を用意し、
「塾の本当の効果」を正しく言い当てられるか試します。
- 隠れた要因（交絡）＝ **家庭の教育熱心さ**（点数にも、塾に通うかにも影響する）
- **塾の本当の効果 ＝ +3点**（この数字を最後に当てられるか？）


In [ ]:
# 練習用の「作りデータ」を作ります（本当の答えが分かっている状態で試すため）
rng = np.random.default_rng(0)
n = 2000  # 生徒2000人ぶん

# ① 家庭の教育熱心さ（今回の"隠れた要因"＝交絡）。人によって高い/低いがある
教育熱心さ = rng.normal(0, 1, n)            # 数値が大きいほど熱心（平均0のまわりにばらつく）

# ② 塾に通うかどうか。教育熱心な家庭ほど塾に通わせやすい、という設定にする
通いやすさ = 1 / (1 + np.exp(-(1.5 * 教育熱心さ)))   # 熱心さが高いほど1に近づく確率
塾に通った = rng.binomial(1, 通いやすさ)             # 1=通った, 0=通っていない

# ③ テストの点数を決める。ここで塾の"本当の効果"を +3点 と仕込んでおく
塾の本当の効果 = 3.0
点数 = (50                            # みんなの基本点
        + 塾の本当の効果 * 塾に通った   # 塾に通うと +3点（← これが今回の正解）
        + 8 * 教育熱心さ              # 家庭が熱心だと大きく上がる（影響がとても大きい）
        + rng.normal(0, 5, n))        # そのほかの偶然のばらつき

# 表（データフレーム）にまとめる
data = pd.DataFrame({
    "家庭の教育熱心さ": 教育熱心さ,
    "塾に通った": 塾に通った,
    "点数": 点数,
})
print("塾に通った生徒:", int(data["塾に通った"].sum()), "人 /", n, "人中")
data.head()


### ステップ1：まず単純に平均点をくらべる
いちばん素朴なやり方＝「塾に通った子」と「通っていない子」で平均点をくらべます。
ここに落とし穴があります。


In [ ]:
# 「塾に通った(1)」「通っていない(0)」ごとに、点数の平均を計算する
平均点 = data.groupby("塾に通った")["点数"].mean()
print("通っていない子の平均点:", round(平均点[0], 1))
print("通った子の平均点    :", round(平均点[1], 1))

# 「通った子の平均」−「通っていない子の平均」＝ 見かけ上の差
見かけの差 = 平均点[1] - 平均点[0]
print("見かけ上の差:", round(見かけの差, 1), "点")
print("→ 塾の本当の効果は +3点のはずなのに、ずっと大きく見える。なぜ？")


### ステップ2：落とし穴の正体（そもそも比べる相手がフェアじゃない）
「塾に通った子」と「通っていない子」は、点数以外の条件も違うかもしれません。
今回いちばん怪しいのは **家庭の教育熱心さ**。これを両者でくらべてみます。


In [ ]:
# 「塾に通った子」「通っていない子」それぞれで、家庭の教育熱心さの平均を出す
熱心さの平均 = data.groupby("塾に通った")["家庭の教育熱心さ"].mean()
print("通っていない子の家庭の熱心さ(平均):", round(熱心さの平均[0], 2))
print("通った子の家庭の熱心さ(平均)    :", round(熱心さの平均[1], 2))
print()
print("→ 塾に通った子の家庭のほうが、もともと教育熱心（数値が高い）。")
print("  つまり点数が高い理由は『塾のおかげ』だけでなく『家庭が熱心だから』も混ざっている。")
print("  この混ざり込みが【交絡】。フェアな比べ方になっていない。")


### ステップ3：フェアにくらべる（似た家庭の子どうしで比べる）
コツは「**塾のこと以外の条件をそろえて**くらべる」こと。
家庭の教育熱心さが近い子どうしをグループにまとめ、その中で塾あり/なしの点差を見ます。


In [ ]:
# 家庭の教育熱心さが近い順に、4つのグループ（低・中低・中高・高）に分ける
data["熱心さグループ"] = pd.qcut(
    data["家庭の教育熱心さ"], 4, labels=["低", "中低", "中高", "高"]
)

# それぞれのグループの中で、「塾に通った子」と「通っていない子」の平均点を出す
表 = data.groupby(["熱心さグループ", "塾に通った"], observed=True)["点数"].mean().unstack()
表["この中での差"] = 表[1] - 表[0]   # 同じような家庭の中での「塾あり − 塾なし」
print(表.round(1))

# 4グループぶんの差を平均する＝家庭の熱心さをそろえたときの塾の効果
似た家庭での差 = 表["この中での差"].mean()
print("\n似た家庭どうしで見た差の平均:", round(似た家庭での差, 1), "点")
print("→ ステップ1の見かけの差より小さくなり、本当の効果 +3 に近づいた！")


### ステップ4：同じことを計算で自動的に（重回帰）
ステップ3では手作業で「似た家庭どうし」をグループにして比べました。
**重回帰**は、これを数式で一気にやってくれる方法です。

点数を、次のような **足し算の式** で表せると考えます：

　点数 ＝ 基本点 ＋（塾の効果）×塾に通った ＋（家庭の効果）×家庭の教育熱心さ

コンピュータは、この式が全生徒のデータに一番よく合うように、
2つの効果（塾の効果・家庭の効果）を **同時に** 求めます。

ここで大事なのは「家庭の教育熱心さ」も式に入れること。こうすると
**「家庭の熱心さが同じだったら、塾に通うと点数が何点変わるか」** が「塾の効果」として出ます。
＝ 家庭の熱心さの影響を **差し引いた（＝調整した）** 塾だけの効果。
これが「**計算で調整**」の意味です（ステップ3の"似た家庭で比べる"を数式で自動化したもの）。


In [ ]:
from sklearn.linear_model import LinearRegression

# 点数 = 基本点 + (塾の効果)×塾に通った + (家庭の効果)×家庭の教育熱心さ
# という式に一番よく合うように、2つの効果を「同時に」求める
説明する列 = data[["塾に通った", "家庭の教育熱心さ"]]
モデル = LinearRegression().fit(説明する列, data["点数"])

基本点   = モデル.intercept_
塾の効果 = モデル.coef_[0]      # 家庭の熱心さをそろえたときの、塾だけの効果
家庭の効果 = モデル.coef_[1]     # 塾をそろえたときの、熱心さ1あたりの効果

print("データに一番合う式が求まりました：")
print(f"  点数 ≒ {基本点:.1f} + {塾の効果:.1f}×塾に通った + {家庭の効果:.1f}×家庭の教育熱心さ")
print()

計算で調整した効果 = 塾の効果
print("→『家庭の熱心さを差し引いた』塾だけの効果:", round(計算で調整した効果, 1), "点")
print("  本当の効果 +3 に近い！（ステップ1の見かけの差 +12 とは大違い）")


### ステップ5：4つの数字を並べて見る
「単純な差」「そろえて比べた差」「本当の効果」を1枚のグラフでくらべます。


In [ ]:
ラベル = ["単純な差\n(落とし穴)", "似た家庭で比較", "計算で調整", "本当の効果"]
値     = [見かけの差, 似た家庭での差, 計算で調整した効果, 塾の本当の効果]
色     = ["gray", "tab:blue", "tab:blue", "tab:red"]

plt.figure(figsize=(7,4))
plt.bar(ラベル, 値, color=色)
plt.axhline(塾の本当の効果, ls="--", color="red")   # 本当の効果の高さに赤い点線
plt.ylabel("塾の効果（点）")
plt.title("条件をそろえて比べると、本当の効果に近づく")
plt.tight_layout(); plt.show()


### この章のまとめ（いちばん大事なこと）
- **「差がある」＝「効果がある」ではない。**
- 差の正体が、くらべる相手の **"もともとの違い"（交絡）** であることが多い。
- 見抜くコツ：「**この2つのグループ、調べたいこと以外の条件はそろってる？**」と自問する。
- 対策：**条件をそろえて比べる**（似た人どうしで比較する／計算で差し引く）。
- 最強の方法：**くじ引きで割り当てる実験（ランダム化）**。誰が塾に通うかを偶然で決めれば、
  家庭の熱心さも自然と両グループで同じくらいになり、フェアに比べられる。
- それでも **測れていない隠れた要因** は残る → 「言えること・言えないこと」を分けて考える。

> 授業での問いかけ例：「このデータだけで"塾に行けば必ず伸びる"と言える？ 何が足りない？」
